In [ ]:
# =============================================================================
# DATA PIPELINE: Multi-scale spatial features + weather + sensor coordinates
# =============================================================================
# Changes from prior version:
#   1. data.rain -> data.weather: now includes temp, humidity, VPD, solar
#      radiation, evapotranspiration, and fuel moisture in addition to precip.
#   2. Added site-level lat/lon, taken from the grid cell with the minimum
#      dist_to_sensor (i.e. the cell physically closest to the water sensor).
#      This is a single static (lat, lon) per site, not a spatial aggregate —
#      it represents "where is this gauge," which is what you want for a
#      regional climate/geology proxy.

LAMBDA_SHORT = 2000    # 2 km: immediate riparian buffer zone dynamics
LAMBDA_MED   = 10000   # 10 km: original baseline structural configuration
LAMBDA_LONG  = 25000   # 25 km: wide-scale regional watershed footprint

crop_cols  = ['Alfalfa', 'Corn', 'Fallow', 'Hay_Pasture', 'Nonag', 'Other', 'Small_Grains', 'Soybeans']
target_col = 'nitrate_con'

# Weather columns available per node-day. precip_in_1d is handled separately
# (it already has bespoke rolling-window logic below); these are the new ones
# to spatially aggregate and add to the daily feature set.
weather_cols = [
    'max_temp', 'min_temp', 'max_rel_humidity', 'min_rel_humidity',
    'vpd', 'solar_rad', 'evapotranspiration', 'fuel_moisture_1000h'
]

all_site_data = []
site_uids = get_site_ids()

print(f"Processing multi-scale features, weather, and coordinates for {len(site_uids)} sites...")

for uid in site_uids:
    site_data = get_data(site_uid=uid)

    df_grid    = site_data.grid.copy()
    df_surplus = site_data.surplus.copy()
    df_crops   = site_data.crops.copy()
    df_weather = site_data.weather.copy()   # renamed from .rain
    df_water   = site_data.water.copy()

    basin_area = getattr(site_data, 'basin_area', df_grid['cell_area'].sum())

    # --- Sensor-proximal coordinates ---
    # Use the grid cell with the smallest dist_to_sensor as the site's
    # representative (lat, lon). This is a single point, not a basin average —
    # it's meant to act as a geographic/climate-zone proxy for the sensor's
    # physical location.
    closest_cell = df_grid.loc[df_grid['dist_to_sensor'].idxmin()]
    site_lat = closest_cell['lat']
    site_lon = closest_cell['lon']

    # --- Spatial Feature Generation across 3 Lambda Scales ---
    for suffix, lam in [('_short', LAMBDA_SHORT), ('_med', LAMBDA_MED), ('_long', LAMBDA_LONG)]:
        df_grid[f'raw_w{suffix}'] = np.exp(-df_grid['dist_to_sensor'] / lam) * df_grid['frac_cell_in_basin']
        total_w = df_grid[f'raw_w{suffix}'].sum()
        df_grid[f'norm_w{suffix}'] = df_grid[f'raw_w{suffix}'] / total_w if total_w > 0 else 0

    # Annual Spatial Features: Surplus N
    df_surplus_w = df_surplus.merge(df_grid[['node_id', 'norm_w_short', 'norm_w_med', 'norm_w_long']], on='node_id')
    df_surplus_w['w_surplus_short'] = df_surplus_w['surplus_kgha'] * df_surplus_w['norm_w_short']
    df_surplus_w['w_surplus_med']   = df_surplus_w['surplus_kgha'] * df_surplus_w['norm_w_med']
    df_surplus_w['w_surplus_long']  = df_surplus_w['surplus_kgha'] * df_surplus_w['norm_w_long']

    annual_surplus = df_surplus_w.groupby('year').agg({
        'w_surplus_short': 'sum', 'w_surplus_med': 'sum', 'w_surplus_long': 'sum'
    }).reset_index()

    # Annual Spatial Features: Crops
    df_crops_w = df_crops.merge(df_grid[['node_id', 'norm_w_short', 'norm_w_med', 'norm_w_long']], on='node_id')
    crop_aggs = {}
    for crop in crop_cols:
        df_crops_w[f'w_{crop}_short'] = df_crops_w[crop] * df_crops_w['norm_w_short']
        df_crops_w[f'w_{crop}_med']   = df_crops_w[crop] * df_crops_w['norm_w_med']
        df_crops_w[f'w_{crop}_long']  = df_crops_w[crop] * df_crops_w['norm_w_long']

        crop_aggs[f'w_{crop}_short'] = 'sum'
        crop_aggs[f'w_{crop}_med']   = 'sum'
        crop_aggs[f'w_{crop}_long']  = 'sum'

    annual_crops    = df_crops_w.groupby('year').agg(crop_aggs).reset_index()
    annual_features = pd.merge(annual_surplus, annual_crops, on='year')

    # --- Daily Weather: Spatial Averaging (basin-area weighted, same as precip) ---
    df_weather = df_weather.merge(df_grid[['node_id', 'frac_cell_in_basin']], on='node_id')
    total_frac = df_grid['frac_cell_in_basin'].sum()

    # data.weather only has 'date' — no year/month/day_of_year/week columns like
    # the old data.rain table did. Derive them here so groupby still works.
    df_weather['date'] = pd.to_datetime(df_weather['date'])
    df_weather['year']        = df_weather['date'].dt.year
    df_weather['month']       = df_weather['date'].dt.month
    df_weather['day_of_year'] = df_weather['date'].dt.dayofyear
    df_weather['week']        = df_weather['date'].dt.isocalendar().week.astype(int)

    df_weather['weighted_precip'] = df_weather['precip_in_1d'] * df_weather['frac_cell_in_basin']
    for col in weather_cols:
        df_weather[f'weighted_{col}'] = df_weather[col] * df_weather['frac_cell_in_basin']

    agg_dict = {'weighted_precip': lambda x: x.sum() / total_frac if total_frac > 0 else 0}
    agg_dict.update({
        f'weighted_{col}': (lambda x: x.sum() / total_frac if total_frac > 0 else 0)
        for col in weather_cols
    })
    # Need first-occurrence date parts; group with named aggs to avoid lambda collisions
    daily_weather = df_weather.groupby('date').agg(
        precip_depth=('weighted_precip', lambda x: x.sum() / total_frac if total_frac > 0 else 0),
        max_temp=('weighted_max_temp', lambda x: x.sum() / total_frac if total_frac > 0 else 0),
        min_temp=('weighted_min_temp', lambda x: x.sum() / total_frac if total_frac > 0 else 0),
        max_rel_humidity=('weighted_max_rel_humidity', lambda x: x.sum() / total_frac if total_frac > 0 else 0),
        min_rel_humidity=('weighted_min_rel_humidity', lambda x: x.sum() / total_frac if total_frac > 0 else 0),
        vpd=('weighted_vpd', lambda x: x.sum() / total_frac if total_frac > 0 else 0),
        solar_rad=('weighted_solar_rad', lambda x: x.sum() / total_frac if total_frac > 0 else 0),
        evapotranspiration=('weighted_evapotranspiration', lambda x: x.sum() / total_frac if total_frac > 0 else 0),
        fuel_moisture_1000h=('weighted_fuel_moisture_1000h', lambda x: x.sum() / total_frac if total_frac > 0 else 0),
        year=('year', 'first'),
        month=('month', 'first'),
        day_of_year=('day_of_year', 'first'),
        week=('week', 'first'),
    ).reset_index()

    # --- Rolling Rainfall Totals (precip only — temp/humidity not rolled by default) ---
    daily_weather = daily_weather.sort_values('date').reset_index(drop=True)
    daily_weather['rain_roll_3d']  = daily_weather['precip_depth'].rolling(window=3,  min_periods=1).sum()
    daily_weather['rain_roll_7d']  = daily_weather['precip_depth'].rolling(window=7,  min_periods=1).sum()
    daily_weather['rain_roll_14d'] = daily_weather['precip_depth'].rolling(window=14, min_periods=1).sum()
    daily_weather['rain_roll_30d'] = daily_weather['precip_depth'].rolling(window=30, min_periods=1).sum()
    daily_weather['rain_roll_60d'] = daily_weather['precip_depth'].rolling(window=60, min_periods=1).sum()
    daily_weather['rain_roll_90d'] = daily_weather['precip_depth'].rolling(window=90, min_periods=1).sum()

    # 7-day rolling mean for slowly-varying weather signals (temp, VPD, ET) —
    # smooths daily noise while preserving seasonal trend. Adjust window as needed.
    for col in ['max_temp', 'min_temp', 'vpd', 'solar_rad', 'evapotranspiration', 'fuel_moisture_1000h']:
        daily_weather[f'{col}_roll_7d'] = daily_weather[col].rolling(window=7, min_periods=1).mean()

    # Merge daily timeline with multi-scale annual features
    daily_features = pd.merge(daily_weather, annual_features, on='year', how='left')

    # --- Target Variable (15-min Nitrate -> Daily Mean) ---
    if 'datetime' in df_water.columns:
        df_water['date'] = pd.to_datetime(df_water['datetime']).dt.date
    else:
        df_water['date'] = pd.to_datetime(df_water.index).date

    daily_target = df_water.groupby('date')[target_col].mean().reset_index()
    daily_target['date'] = pd.to_datetime(daily_target['date'])
    daily_features['date'] = pd.to_datetime(daily_features['date'])

    site_combined = pd.merge(daily_target, daily_features, on='date', how='inner')
    site_combined['site_uid']    = uid
    site_combined['basin_area']  = basin_area
    site_combined['site_lat']    = site_lat
    site_combined['site_lon']    = site_lon

    site_combined['site_historical_mean_rain'] = daily_weather['precip_depth'].mean()

    all_site_data.append(site_combined)

# Combine all processed datasets
df_master_raw = pd.concat(all_site_data, ignore_index=True).sort_values('date')

# =============================================================================
# DATA CLEANING & FEATURE LIST
# =============================================================================
feature_cols = [
    'month', 'day_of_year', 'week', 'basin_area', 'site_historical_mean_rain',
    'site_lat', 'site_lon',
    'precip_depth', 'rain_roll_3d', 'rain_roll_7d', 'rain_roll_14d',
    'rain_roll_30d', 'rain_roll_60d', 'rain_roll_90d',
    'max_temp', 'min_temp', 'max_rel_humidity', 'min_rel_humidity',
    'vpd', 'solar_rad', 'evapotranspiration', 'fuel_moisture_1000h',
    'max_temp_roll_7d', 'min_temp_roll_7d', 'vpd_roll_7d',
    'solar_rad_roll_7d', 'evapotranspiration_roll_7d', 'fuel_moisture_1000h_roll_7d',
]

for suffix in ['_short', '_med', '_long']:
    feature_cols.append(f'w_surplus{suffix}')
    for crop in crop_cols:
        feature_cols.append(f'w_{crop}{suffix}')

# 1. Drop rows missing target data
df_master = df_master_raw.dropna(subset=[target_col]).copy()
df_master = df_master[np.isfinite(df_master[target_col])]

# 2. Impute and clean feature matrix
for col in feature_cols:
    df_master[col] = df_master[col].replace([np.inf, -np.inf], np.nan)
    if df_master[col].isnull().any():
        if 'rain' in col or 'precip' in col:
            df_master[col] = df_master[col].fillna(0)
        elif col in ('site_lat', 'site_lon'):
            # Static per-site value — fill from the site's own non-null value
            df_master[col] = df_master.groupby('site_uid')[col].transform(lambda x: x.fillna(x.iloc[0]))
        else:
            df_master[col] = df_master.groupby('site_uid')[col].transform(lambda x: x.fillna(x.median()))

# 3. Anomaly Management: Isolate severe regional spatial outliers (>30 mg/L)
site_nitrate_averages = df_master.groupby('site_uid')[target_col].mean()
outlier_uids = site_nitrate_averages[site_nitrate_averages > 30.0].index.tolist()

if outlier_uids:
    print(f"Isolating extreme spatial outlier site(s) from standard pool: {outlier_uids}")
    df_modeling_pool = df_master[~df_master['site_uid'].isin(outlier_uids)].copy()
else:
    df_modeling_pool = df_master.copy()

print(f"\nData matrix compiled successfully.")
print(f" -> Cleaned rows available for modeling: {df_modeling_pool.shape[0]}")
print(f" -> Feature count: {len(feature_cols)} (added {len(weather_cols)} weather vars + 2 coordinate cols)")

# Save for downstream classification script
df_master_raw.to_parquet('df_master_raw_hydrology.parquet', index=False)
print("Saved df_master_raw_hydrology.parquet")

In [ ]:
import sys
sys.path.insert(0, "../")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, auc
from sklearn.model_selection import ParameterGrid
from xgboost import XGBClassifier

# =============================================================================
# CONFIGURATION
# =============================================================================
EPA_THRESHOLD = 10.0   # mg/L EPA maximum contaminant level for nitrate
RANDOM_STATE  = 42
K_CLUSTERS    = 6      # geographic clusters; 1 held out as final test (~17%)

# Surplus data is only available through 2017. Records after 2017 have surplus
# features set to zero in the raw data — we add a binary flag so the model
# can distinguish genuine zero surplus from missing data.
SURPLUS_CUTOFF = 2017

print("=" * 60)
print("NITRATE BREACH CLASSIFIER")
print(f"  EPA threshold : {EPA_THRESHOLD} mg/L")
print(f"  Spatial CV    : Leave-One-Cluster-Out ({K_CLUSTERS} clusters)")
print(f"  Holdout       : 1 held-out geographic cluster (~17% of sites)")
print(f"  Surplus flag  : records post-{SURPLUS_CUTOFF} flagged as unobserved")
print("=" * 60)

# =============================================================================
# 1. LOAD DATA
# =============================================================================
df = pd.read_parquet('df_master_raw_hydrology.parquet')
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

id_col = 'site_uid' if 'site_uid' in df.columns else 'site_id'

print(f"\nLoaded {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Sites      : {df[id_col].nunique()}")

# =============================================================================
# 2. SURPLUS OBSERVED FLAG
# =============================================================================
# Surplus data is available only through SURPLUS_CUTOFF. Rather than letting
# the model treat post-cutoff zeros as genuine agronomic signal, we add an
# explicit binary flag so it can learn to discount surplus features when the
# data is unobserved. This is the honest encoding of the missing-data regime.
surplus_cols = [c for c in df.columns if 'surplus' in c.lower()]
df['surplus_observed'] = (df['year'] <= SURPLUS_CUTOFF).astype(int)

print(f"\nSurplus flag added:")
print(f"  Surplus cols flagged : {surplus_cols}")
print(f"  Observed (≤{SURPLUS_CUTOFF})     : {df['surplus_observed'].sum():,} rows")
print(f"  Unobserved (>{SURPLUS_CUTOFF})   : {(df['surplus_observed']==0).sum():,} rows")

# =============================================================================
# 3. BINARY TARGET
# =============================================================================
df['nitrate_breach'] = (df['nitrate_con'] > EPA_THRESHOLD).astype(int)
breach_rate = df['nitrate_breach'].mean()
print(f"\nBreach prevalence : {breach_rate:.2%}")
print(f"Random-guess PR-AUC baseline : {breach_rate:.4f}")

# =============================================================================
# 4. FEATURE ENGINEERING
# =============================================================================
def engineer_features(df_in):
    df_out = df_in.copy()

    # --- Cyclic temporal encoding ---
    # Raw integers are ordinal not cyclic — Dec(12) and Jan(1) appear maximally
    # distant when they are adjacent. Sin/cos wraps the calendar correctly.
    # Raw integers are also retained: they capture monotone seasonal trends
    # that the trig pair alone cannot represent.
    for col, period in [('month', 12), ('day_of_year', 365), ('week', 52)]:
        if col in df_out.columns:
            df_out[f'{col}_sin'] = np.sin(2 * np.pi * df_out[col] / period)
            df_out[f'{col}_cos'] = np.cos(2 * np.pi * df_out[col] / period)

    # --- Seasonal interaction features ---
    # Fertilizer application peaks Apr–May; spring rain flushes nitrate into
    # tile drains. These cross-products encode the dominant physical loading
    # mechanism explicitly rather than requiring deep nested splits.
    df_out['spring_flag'] = df_out['month'].isin([3, 4, 5]).astype(int)
    df_out['summer_flag'] = df_out['month'].isin([6, 7, 8]).astype(int)
    df_out['spring_rain_7d']   = df_out['spring_flag'] * df_out['rain_roll_7d']
    df_out['spring_rain_30d']  = df_out['spring_flag'] * df_out['rain_roll_30d']
    if 'w_Corn_short' in df_out.columns:
        df_out['spring_corn_rain'] = (
            df_out['spring_flag'] * df_out['w_Corn_short'] * df_out['rain_roll_7d']
        )

    # --- Basin area ---
    # Sites span four orders of magnitude — log compresses the range so the
    # model splits on hydrology rather than raw scale.
    if 'basin_area' in df_out.columns:
        df_out['log_basin_area']   = np.log1p(df_out['basin_area'])
        df_out['volume_proxy_30d'] = df_out['rain_roll_30d'] * df_out['basin_area']
        df_out['volume_proxy_90d'] = df_out['rain_roll_90d'] * df_out['basin_area']

    # --- Leaching risk ---
    # High rainfall + low evapotranspiration = maximum nitrate flush through
    # tile drains. This ratio gives the model a direct handle on the
    # rain-vs-uptake competition that drives leaching events.
    if {'evapotranspiration', 'rain_roll_7d'}.issubset(df_out.columns):
        df_out['leaching_risk_7d'] = (
            df_out['rain_roll_7d'] / (df_out['evapotranspiration'] + 1.0)
        )

    return df_out

df = engineer_features(df)
print("\nFeature engineering complete.")

# =============================================================================
# 5. SPATIAL CLUSTER SPLIT — HOLDOUT + CV FOLDS
# =============================================================================
# K-Means clusters sites by lat/lon into K geographic regions. One cluster
# is held out as the final test set — locked away until Section 11. The
# remaining K-1 clusters form the development pool and are used for nested
# spatial CV (Leave-One-Cluster-Out within the dev pool).
#
# Why spatial clusters instead of temporal or conflict-graph splits:
#   - Temporal: only 3-4 years of training data per site (most start 2014),
#     making a temporal holdout severely data-starved for training.
#   - Conflict graph: over-groups Iowa sites into a 34-site mega-component
#     (Des Moines system), making balanced CV mathematically impossible.
#   - Spatial clusters: transparent, balanced, reproducible, and tests the
#     scientifically meaningful question of geographic generalisation.
#
# Holdout cluster selection: the cluster geographically closest to the
# centroid of all Iowa sites — most representative of the general
# distribution, not an edge-case outlier region.

site_coords = (
    df.groupby(id_col)[['site_lat', 'site_lon']]
    .first()
    .dropna()
    .reset_index()
)

_scaler        = StandardScaler()
_coords_scaled = _scaler.fit_transform(site_coords[['site_lat', 'site_lon']])
_kmeans        = KMeans(n_clusters=K_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
site_coords['cluster'] = _kmeans.fit_predict(_coords_scaled)

# Select holdout cluster: closest to overall centroid
_all_centroid         = _coords_scaled.mean(axis=0)
_cluster_centroids    = np.array([
    _coords_scaled[site_coords['cluster'] == c].mean(axis=0)
    for c in range(K_CLUSTERS)
])
_dists               = np.linalg.norm(_cluster_centroids - _all_centroid, axis=1)
holdout_cluster      = int(np.argmin(_dists))

holdout_sites = site_coords.loc[site_coords.cluster == holdout_cluster, id_col].tolist()
dev_sites     = site_coords.loc[site_coords.cluster != holdout_cluster, id_col].tolist()

df_dev     = df[df[id_col].isin(dev_sites)].copy()
df_holdout = df[df[id_col].isin(holdout_sites)].copy()

print(f"\nSpatial cluster holdout split ({K_CLUSTERS} clusters):")
print(f"  Holdout cluster   : {holdout_cluster} "
      f"({len(holdout_sites)} sites, {len(df_holdout):,} rows) ← locked")
print(f"  Development pool  : {len(dev_sites)} sites, {len(df_dev):,} rows")
print(f"  Holdout fraction  : {len(df_holdout)/len(df):.1%} of rows, "
      f"{len(holdout_sites)/len(site_coords):.1%} of sites")
print(f"  *** df_holdout is now locked — not touched until Section 11 ***")

# =============================================================================
# 6. DEFINE FEATURE COLUMNS
# =============================================================================
# Excluded from features:
#   - id_col, date, year, cluster : identifiers / CV scaffolding
#   - nitrate_con, nitrate_breach  : target and raw target
#   - basin_area                   : replaced by log_basin_area
#
# Included deliberately:
#   - site_lat, site_lon  : legitimate regional proxies for geology and
#     climate zone. With spatial cluster CV the held-out cluster is
#     geographically distinct, so coordinates cannot act as a lookup key
#     for target values — they carry real regional signal.
#   - surplus_observed    : tells the model when surplus data is missing
#     rather than genuinely zero.
#   - All weather columns  : flow through automatically.

drop_cols = {
    id_col, 'date', 'year', 'cluster',
    'nitrate_con', 'nitrate_breach',
    'basin_area',   # replaced by log_basin_area
}

feature_cols = [c for c in df_dev.columns if c not in drop_cols]

# Sanity checks
geo_cols     = [c for c in feature_cols if c in ('site_lat', 'site_lon')]
weather_cols = [c for c in feature_cols if c in (
    'max_temp', 'min_temp', 'max_rel_humidity', 'min_rel_humidity',
    'vpd', 'solar_rad', 'evapotranspiration', 'fuel_moisture_1000h',
    'max_temp_roll_7d', 'min_temp_roll_7d', 'vpd_roll_7d',
    'solar_rad_roll_7d', 'evapotranspiration_roll_7d',
    'fuel_moisture_1000h_roll_7d',
)]

print(f"\nFeature matrix : {len(feature_cols)} features")
print(f"  Coordinates  : {geo_cols}")
print(f"  Weather      : {len(weather_cols)} columns")
print(f"  Surplus flag : {'surplus_observed' in feature_cols}")

# Ensure holdout has exactly the same feature columns
missing_in_holdout = set(feature_cols) - set(df_holdout.columns)
if missing_in_holdout:
    print(f"WARNING: features missing in holdout: {missing_in_holdout}")

# =============================================================================
# 7. CV FOLD ASSIGNMENTS (Leave-One-Cluster-Out on dev pool)
# =============================================================================
dev_coords   = site_coords[site_coords[id_col].isin(dev_sites)].copy()
_dev_clusters = sorted(dev_coords['cluster'].unique())
NUM_FOLDS     = len(_dev_clusters)   # K-1 = 5 folds
_c2f          = {c: i for i, c in enumerate(_dev_clusters)}
dev_coords['fold'] = dev_coords['cluster'].map(_c2f)

outer_folds = dev_coords.set_index(id_col)[['cluster', 'fold']].rename(
    columns={'cluster': 'group'}
)
outer_folds.index.name = id_col

print(f"\nCV strategy : Leave-One-Cluster-Out ({NUM_FOLDS} folds on dev pool)")
print(f"{'Fold':>5}  {'Val sites':>9}  {'Val%':>6}  {'Train sites':>11}"
      f"  {'Val rows':>9}  {'Train rows':>11}")
print(f"  {'─'*60}")
for _f in range(NUM_FOLDS):
    _vs = outer_folds.index[outer_folds.fold == _f].tolist()
    _ts = outer_folds.index[outer_folds.fold != _f].tolist()
    _vr = df_dev[df_dev[id_col].isin(_vs)].shape[0]
    _tr = df_dev[df_dev[id_col].isin(_ts)].shape[0]
    print(f"{_f+1:>5}  {len(_vs):>9}  {len(_vs)/len(outer_folds)*100:>5.1f}%"
          f"  {len(_ts):>11}  {_vr:>9,}  {_tr:>11,}")

# =============================================================================
# 8. HYPERPARAMETER GRID
# =============================================================================
PARAM_GRID = list(ParameterGrid({
    'max_depth':        [4, 6],
    'learning_rate':    [0.03, 0.05],
    'n_estimators':     [600],          # ceiling; early stopping fires before
    'subsample':        [0.8],
    'colsample_bytree': [0.8],
    'reg_alpha':        [0.0, 0.5, 1.0],   # L1: sparse feature selection
    'reg_lambda':       [1.0, 2.0],         # L2: leaf weight shrinkage
    'gamma':            [0.0, 0.5],         # min loss reduction to split
    'min_child_weight': [3, 10],
}))
print(f"\nGrid search : {len(PARAM_GRID)} configurations")
print(f"Total fits  : {len(PARAM_GRID)} × {NUM_FOLDS-1} inner × {NUM_FOLDS} outer"
      f" = {len(PARAM_GRID) * (NUM_FOLDS-1) * NUM_FOLDS} fits")

# =============================================================================
# 9. NESTED SPATIAL CV
# =============================================================================
plt.figure(figsize=(9, 7))
mean_recall         = np.linspace(0, 1, 100)
precisions_all      = []
fold_auc_scores     = []
best_params_by_fold = []

print(f"\n{'='*60}")
print(f"Starting {NUM_FOLDS}-Fold Nested Spatial CV")
print(f"{'='*60}\n")

for fold_idx in range(NUM_FOLDS):
    print(f"--- Fold {fold_idx+1}/{NUM_FOLDS} ---")

    val_sites   = outer_folds.index[outer_folds.fold == fold_idx].tolist()
    train_sites = outer_folds.index[outer_folds.fold != fold_idx].tolist()

    df_tr = df_dev[df_dev[id_col].isin(train_sites)].copy()
    df_va = df_dev[df_dev[id_col].isin(val_sites)].copy()

    print(f"  Sites — train: {len(train_sites):>3} "
          f"({len(train_sites)/len(outer_folds)*100:.1f}%)  "
          f"val: {len(val_sites):>3} "
          f"({len(val_sites)/len(outer_folds)*100:.1f}%)")
    print(f"  Rows  — train: {len(df_tr):>6,}  val: {len(df_va):>6,}")

    X_va = df_va[feature_cols]
    y_va = df_va['nitrate_breach']

    # --- Inner CV: hold out one of the remaining clusters ---
    _inner = outer_folds[outer_folds.index.isin(train_sites)].copy()
    _iclusters = sorted(_inner['fold'].unique())
    _iremap    = {c: i for i, c in enumerate(_iclusters)}
    _inner['inner_fold'] = _inner['fold'].map(_iremap)
    N_INNER = len(_iclusters)

    best_inner_auc = -1.0
    best_params    = PARAM_GRID[0]

    for params in PARAM_GRID:
        inner_aucs = []

        for ii in range(N_INNER):
            i_val   = _inner.index[_inner.inner_fold == ii].tolist()
            i_train = _inner.index[_inner.inner_fold != ii].tolist()

            df_it = df_tr[df_tr[id_col].isin(i_train)]
            df_iv = df_tr[df_tr[id_col].isin(i_val)]

            X_it, y_it = df_it[feature_cols], df_it['nitrate_breach']
            X_iv, y_iv = df_iv[feature_cols], df_iv['nitrate_breach']

            if y_it.nunique() < 2 or y_iv.nunique() < 2:
                continue

            # Downsample majority class 2:1 on training split only
            idx_neg = y_it[y_it == 0].index
            idx_pos = y_it[y_it == 1].index
            n_samp  = min(len(idx_neg), len(idx_pos) * 2)
            rng     = np.random.default_rng(RANDOM_STATE)
            bal_idx = np.concatenate([idx_pos,
                                      rng.choice(idx_neg, n_samp, replace=False)])
            X_bal, y_bal = X_it.loc[bal_idx], y_it.loc[bal_idx]
            imb = (y_bal == 0).sum() / max(1, (y_bal == 1).sum())

            m = XGBClassifier(
                objective='binary:logitraw',
                eval_metric='aucpr',
                early_stopping_rounds=25,
                tree_method='hist',
                n_jobs=-1,
                random_state=RANDOM_STATE,
                scale_pos_weight=imb,
                **params
            )
            m.fit(X_bal, y_bal, eval_set=[(X_iv, y_iv)], verbose=False)

            sc = m.predict(X_iv, output_margin=True)
            p, r, _ = precision_recall_curve(y_iv, sc)
            inner_aucs.append(auc(r, p))

        if inner_aucs and np.mean(inner_aucs) > best_inner_auc:
            best_inner_auc = np.mean(inner_aucs)
            best_params    = params

    best_params_by_fold.append(best_params)
    print(f"  Best inner PR-AUC : {best_inner_auc:.4f}")
    print(f"  Best params       : {best_params}")

    # Retrain on full outer training set with winning params
    X_tr = df_tr[feature_cols]
    y_tr = df_tr['nitrate_breach']

    idx_neg = y_tr[y_tr == 0].index
    idx_pos = y_tr[y_tr == 1].index
    n_samp  = min(len(idx_neg), len(idx_pos) * 2)
    rng     = np.random.default_rng(RANDOM_STATE)
    bal_idx = np.concatenate([idx_pos,
                              rng.choice(idx_neg, n_samp, replace=False)])
    X_bal_f = X_tr.loc[bal_idx]
    y_bal_f = y_tr.loc[bal_idx]
    imb = (y_bal_f == 0).sum() / max(1, (y_bal_f == 1).sum())

    fold_model = XGBClassifier(
        objective='binary:logitraw',
        eval_metric='aucpr',
        early_stopping_rounds=30,
        tree_method='hist',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        scale_pos_weight=imb,
        **best_params
    )
    fold_model.fit(X_bal_f, y_bal_f, eval_set=[(X_va, y_va)], verbose=False)

    sc_va = fold_model.predict(X_va, output_margin=True)
    p_v, r_v, _ = precision_recall_curve(y_va, sc_va)
    fold_auc = auc(r_v, p_v)
    fold_auc_scores.append(fold_auc)

    plt.plot(r_v, p_v, alpha=0.3, linestyle='--',
             label=f'Fold {fold_idx+1} (AUC={fold_auc:.4f})')
    precisions_all.append(np.interp(mean_recall, r_v[::-1], p_v[::-1]))
    print(f"  Outer PR-AUC      : {fold_auc:.4f}\n")

# =============================================================================
# 10. CV RESULTS & PLOTS
# =============================================================================
mean_auc      = np.mean(fold_auc_scores)
std_auc       = np.std(fold_auc_scores)
mean_prec     = np.mean(precisions_all, axis=0)
mean_curve_auc = auc(mean_recall, mean_prec)

print(f"{'='*60}")
print(f"CV RESULTS  (spatial generalisation, all years)")
print(f"  Mean PR-AUC : {mean_auc:.4f} ± {std_auc:.4f}")
print(f"  Per-fold    : {[f'{x:.4f}' for x in fold_auc_scores]}")
print(f"  Baseline    : {breach_rate:.4f}  (random guess)")
print(f"{'='*60}\n")

plt.plot(mean_recall, mean_prec, color='firebrick', lw=2.5,
         label=f'Mean PR-AUC = {mean_curve_auc:.4f} ± {std_auc:.4f}')
plt.axhline(y=breach_rate, color='gray', linestyle=':', lw=1.5,
            label=f'Random Guess ({breach_rate:.3f})')
plt.xlim([-0.02, 1.02]); plt.ylim([-0.02, 1.02])
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Spatial CV Precision-Recall Curve\n'
          f'Leave-One-Cluster-Out ({NUM_FOLDS} geographic folds)',
          fontsize=13, fontweight='bold')
plt.legend(loc='lower left', fontsize=9)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('pr_curve_spatial_cv.png', dpi=150)
plt.show()

# --- Feature importance (last fold model) ---
importances = pd.Series(
    fold_model.feature_importances_, index=feature_cols
).sort_values(ascending=False)

print("Top 20 Feature Importances (last CV fold):")
print(importances.head(20).to_string())

fig, ax = plt.subplots(figsize=(9, 6))
importances.head(20).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('Top 20 Feature Importances — Nitrate Breach Classifier',
             fontweight='bold')
ax.set_xlabel('XGBoost Feature Importance (Gain)')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print("\nBest hyperparameters per CV fold:")
for i, p in enumerate(best_params_by_fold):
    print(f"  Fold {i+1}: {p}")

# =============================================================================
# 11. FINAL MODEL — RETRAIN ON FULL DEV SET
# =============================================================================
# Majority-vote hyperparameters across all CV folds — more stable than
# picking the single best-performing fold's params.
param_keys        = list(best_params_by_fold[0].keys())
best_final_params = {
    k: Counter(p[k] for p in best_params_by_fold).most_common(1)[0][0]
    for k in param_keys
}
print(f"\nMajority-vote params for deployment model: {best_final_params}")

X_dev_all = df_dev[feature_cols]
y_dev_all = df_dev['nitrate_breach']

idx_neg = y_dev_all[y_dev_all == 0].index
idx_pos = y_dev_all[y_dev_all == 1].index
n_samp  = min(len(idx_neg), len(idx_pos) * 2)
rng     = np.random.default_rng(RANDOM_STATE)
bal_idx = np.concatenate([idx_pos,
                          rng.choice(idx_neg, n_samp, replace=False)])
X_bal_dev = X_dev_all.loc[bal_idx]
y_bal_dev = y_dev_all.loc[bal_idx]
imb = (y_bal_dev == 0).sum() / max(1, (y_bal_dev == 1).sum())

# Early stopping monitor: fold 0 val set — never the holdout
_es_sites = outer_folds.index[outer_folds.fold == 0].tolist()
_df_es    = df_dev[df_dev[id_col].isin(_es_sites)]
X_es, y_es = _df_es[feature_cols], _df_es['nitrate_breach']

deployment_model = XGBClassifier(
    objective='binary:logitraw',
    eval_metric='aucpr',
    early_stopping_rounds=30,
    tree_method='hist',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    scale_pos_weight=imb,
    **best_final_params
)
deployment_model.fit(X_bal_dev, y_bal_dev,
                     eval_set=[(X_es, y_es)], verbose=False)
print("Deployment model trained on full development set.")

# =============================================================================
# 12. HOLDOUT EVALUATION  (done exactly once)
# =============================================================================
# The holdout cluster was locked away in Section 5 and has had zero influence
# on feature engineering, CV, hyperparameter selection, or model training.
# This is the single unbiased deployment performance estimate.

X_ho = df_holdout[feature_cols]
y_ho = df_holdout['nitrate_breach']
ho_breach_rate = y_ho.mean()

sc_ho      = deployment_model.predict(X_ho, output_margin=True)
p_ho, r_ho, _ = precision_recall_curve(y_ho, sc_ho)
holdout_auc = auc(r_ho, p_ho)

print(f"\n{'='*60}")
print(f"HOLDOUT RESULTS  (cluster {holdout_cluster} — never seen during training)")
print(f"  Sites             : {df_holdout[id_col].nunique()}")
print(f"  Records           : {len(y_ho):,}")
print(f"  Breach prevalence : {ho_breach_rate:.2%}")
print(f"  Holdout PR-AUC    : {holdout_auc:.4f}")
print(f"  CV mean PR-AUC    : {mean_auc:.4f} ± {std_auc:.4f}")
print(f"  Gap (CV − Holdout): {mean_auc - holdout_auc:+.4f}")
if abs(mean_auc - holdout_auc) < 0.04:
    print("  → CV well-calibrated — gap within noise.")
elif mean_auc > holdout_auc:
    print("  → CV optimistic — model may have subtly overfit to dev geography.")
else:
    print("  → Holdout exceeds CV — holdout cluster may be easier to predict.")
print(f"{'='*60}")

# --- Final comparison plot ---
fig, ax = plt.subplots(figsize=(9, 7))
for i, (p_f, auc_f) in enumerate(zip(precisions_all, fold_auc_scores)):
    ax.plot(mean_recall, p_f, alpha=0.2, linestyle='--',
            label=f'CV Fold {i+1} ({auc_f:.3f})')
ax.plot(mean_recall, mean_prec, color='steelblue', lw=2.5,
        label=f'CV Mean = {mean_curve_auc:.4f} ± {std_auc:.4f}')
ax.plot(r_ho, p_ho, color='firebrick', lw=2.5,
        label=f'Holdout (cluster {holdout_cluster}) = {holdout_auc:.4f}')
ax.axhline(y=breach_rate, color='gray', linestyle=':', lw=1.5,
           label=f'Random Guess ({breach_rate:.3f})')
ax.set_xlim([-0.02, 1.02]); ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax.set_ylabel('Precision (Reliability)', fontsize=12)
ax.set_title('Nitrate Breach Classifier — CV vs Geographic Holdout\n'
             f'Spatial Leave-One-Cluster-Out | {len(feature_cols)} features',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower left', fontsize=8)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('pr_curve_final.png', dpi=150)
plt.show()